In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# pip install torchvision
# 

In [3]:
import os,sys
#sys.path.append('/work/qdiff/mo_utils')
sys.executable


'/home/nadavg/phase2-sdk/sdk_virtualenv/bin/python'

In [4]:
print(os.getcwdb())
os.chdir('/home/nadavg/q-diffusion')
print(os.getcwdb())

b'/home/nadavg/q-diffusion/scripts/hf15'
b'/home/nadavg/q-diffusion'


In [5]:
os.environ['CUDA_VISIBLE_DEVICES'] = '4'

In [6]:
import torch
torch.cuda.is_available()

False

In [38]:
from mo_utils.utils.stand_alone_utils.har_utils import get_har_files,get_params_from_har
from mo_utils.utils.stand_alone_utils.pytorch2accelras import (
    get_nested_attr,
    get_weight_and_bias_from_layer_name,
    acc_ker_to_pytorch_weight,
    UpdateUnet,
    )
from mo_utils.utils.stand_alone_utils.quant_utils import calc_snr,calc_stats
from mo_utils.utils.har_utils import super_netron


In [8]:
from src.utils.torch_utils import add_full_name_to_module
from scripts.hf15.init_pipe import init_pipe

/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/flax/struct.py:136: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/flax/struct.py:136: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
2025-03-31 11:20:24.546500: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/flax/struct.py:136: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, k

In [9]:
import netron
from pathlib import Path
from collections import namedtuple

In [10]:
import torch
import torch.nn as nn
from diffusers import StableDiffusionPipeline
from scripts.hf15.txt2img import get_train_samples
from qdiff.quant_model import QuantModel

/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/transformers/models/clip/feature_extraction_clip.py:28: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(


In [11]:
cali_data_path ='/genai/users/nadavg/sd/qdiff_hf15_verb/gen_calib/calib_dict_steps20.pt' 
sample_data = torch.load(cali_data_path)
dummy_args = namedtuple('Args',['cali_n','cali_st','custom_steps','cond'])
opt = dummy_args(1,20,None,True)
cali_data = get_train_samples(opt, sample_data,20)
cali_xs, cali_ts, cali_cs = cali_data

In [12]:
pipe = init_pipe()
unet = pipe.unet
add_full_name_to_module(unet)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, plea

In [13]:
tb =get_nested_attr(unet,'down_blocks.0.attentions.0.transformer_blocks.0',ignore_last=0,
                       spliter='.',)

In [14]:
type(tb)

diffusers.models.attention.BasicTransformerBlock

In [15]:
attn = get_nested_attr(unet,'down_blocks.0.attentions.0.transformer_blocks.0.attn2',ignore_last=0,
                       spliter='.',)
attn.full_name

'down_blocks.0.attentions.0.transformer_blocks.0.attn2'

In [16]:
ascale = attn.scale

In [17]:
ascale,ascale**0.5

(0.15811388300841897, 0.3976353643835253)

In [18]:
attn.query_dim, attn.heads

(320, 8)

In [19]:
norm = get_nested_attr(unet,
                       'down_blocks.0.attentions.0.transformer_blocks.0.norm2',
                       ignore_last=0,spliter='.',)
norm.full_name

'down_blocks.0.attentions.0.transformer_blocks.0.norm2'

In [20]:
from src.utils.torch_utils import save_input_hook

In [21]:
hookn = norm.register_forward_hook(save_input_hook)
hookq=attn.to_k.register_forward_hook(save_input_hook)


In [22]:
ind_batch= slice(30,31)
out_org = unet(cali_xs[ind_batch],cali_ts[ind_batch],cali_cs[ind_batch])
out_org[0].shape

torch.Size([1, 4, 64, 64])

In [23]:
norm.saved_inputs[0][0].shape,attn.to_k.saved_inputs[0][0].shape

(torch.Size([1, 4096, 320]), torch.Size([1, 77, 768]))

In [24]:
inp = [norm.saved_inputs[0][0].detach().clone() ,attn.to_k.saved_inputs[0][0].detach().clone()]
inp[0].shape,inp[1].shape

(torch.Size([1, 4096, 320]), torch.Size([1, 77, 768]))

In [25]:
class normatt(nn.Module):
    def __init__(self,norm,attn):
        super().__init__()
        self.norm2 = norm
        self.attn2 = attn
    def forward(self,x,context):
        x = self.norm2(x)
        x = self.attn2(x,context)
        return x

In [26]:
#inp = torch.randn(1,4096,320)
na2 = normatt(norm,attn)

In [27]:
out_org = na2(*inp)
out_org.shape

torch.Size([1, 4096, 320])

In [31]:
inp[0].shape,inp[1].shape

(torch.Size([1, 4096, 320]), torch.Size([1, 77, 768]))

In [41]:
unet_model_onnx_path = "normattn2.onnx"
torch.onnx.export(na2, {'x':inp[0],'context':inp[1]}, unet_model_onnx_path, 
                   output_names=["output"],
                    input_names=["x","context"],
                     dynamic_axes={
                    "x": {0: "batch"},
                        "context": {0: "batch"},
            "output": {0: "batch"},
        },
)

In [42]:
! ls normattn2.onnx

normattn2.onnx


In [43]:
! onnxsim normattn2.onnx normattn2.sim.onnx --overwrite-input-shape x:1,4096,320 context:1,77,768


Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                    ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add                │ 1              │ 1                │
│ Cast               │ 5              │ 0                │
│ Concat             │ 4              │ 0                │
│ Constant           │ 30             │ 10               │
│ Div                │ 3              │ 0                │
│ Gather             │ 2              │ 0                │
│ LayerNormalization │ 1              │ 1                │
│ MatMul             │ 6              │ 6                │
│ Mul                │ 3              │ 2                │
│ Reshape            │ 4              │ 4                │
│ Shape              │ 3              │ 0                │
│ Slice              │ 1              │ 0                │
│ Softmax            │ 1              │ 1                │
│ Sqrt   

In [40]:
super_netron('normattn2.sim.onnx',port=8096)

Serving 'normattn2.sim.onnx' at http://localhost:8096


In [44]:
! hailo parser onnx normattn2.sim.onnx --start-node-names x context

[info] Current Time: 11:49:02, 03/31/25
[info] CPU: Architecture: x86_64, Model: AMD EPYC 9555 64-Core Processor, Number Of Cores: 256, Utilization: 3.2%
[info] Memory: Total: 1007GB, Available: 928GB
[info] System info: OS: Linux, Kernel: 6.5.0-27-generic
[info] Hailo DFC Version: 3.32.0.dev0
[info] HailoRT Version: 4.21.0
[info] PCIe: No Hailo PCIe device was found
[info] Running `hailo parser onnx normattn2.sim.onnx --start-node-names x context`
[info] Found a '.' character in net_name, which isn't supported. New net_name is normattn2_sim
[info] Translation started on ONNX model normattn2_sim
[info] Restored ONNX model normattn2_sim (completion time: 00:00:00.01)
[info] Extracted ONNXRuntime meta-data for Hailo model (completion time: 00:00:00.02)
[info] Start nodes mapped from original model: 'x': 'normattn2_sim/input_layer1', 'context': 'normattn2_sim/input_layer2'.
[info] End nodes mapped from original model: '/attn2/to_out.0/Add'.
[info] Translation completed on ONNX model norma

In [45]:
from mo_utils.utils.har_utils import super_netron

In [46]:
super_netron('normattn2_sim.har')

all_names=['normattn2_sim.hn', 'normattn2_sim.npz', 'normattn2_sim.original_model_meta.json', 'normattn2_sim.metadata.json']
loading  params_name='normattn2_sim.hn' ...
Serving 'temp.hn' at http://localhost:19226


In [47]:
from hailo_sdk_client.runner.client_runner import ClientRunner

In [48]:
cr = ClientRunner(har='normattn2_sim.har',)

In [49]:
cr.optimize_full_precision(calib_data=inp)

In [50]:
cr.save_har('normattn2_sim_fp.har')

[info] Saved HAR to: /home/nadavg/q-diffusion/normattn2_sim_fp.har


In [26]:
from mo_utils.utils.acceleras_utils_verb import AccModel 
from hailo_sdk_client import InferenceContext


ac_na = AccModel(har_path='normattn_sim_fp.har',inf_context=InferenceContext.SDK_FP_OPTIMIZED)

warning! no input shape sp model mot build. 
 run inference before using the model


2025-03-30 14:14:03.134090: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.


In [34]:
inp.shape
#inp.unsqueeze(0).shape

torch.Size([1, 4096, 320])

In [ ]:
out_ac = ac_na.model(inp.unsqueeze(0).numpy())

In [49]:
inter_res = ac_na.get_inter_reses_full(
    model=ac_na.model,
    input=inp.unsqueeze(0).numpy(),
    only_native_res=True
    )

In [52]:
inter_res['native']['normattn_sim/layer_normalization1'][0].shape

TensorShape([1, 1, 4096, 320])

In [33]:
calc_snr(out_ac[0],out_org[0])

66.76759222372209

In [34]:
super_netron('normattn_sim_fp.har')

all_names=['normattn_sim.hn', 'normattn_sim.native.hn', 'normattn_sim.npz', 'normattn_sim.fpo.npz', 'normattn_sim.stats.npz', 'normattn_sim.original_model_meta.json', 'normattn_sim.modifications_meta_data.json', 'normattn_sim.modification_params.npz', 'normattn_sim.metadata.json']
loading  params_name='normattn_sim.hn' ...
Serving 'temp.hn' at http://localhost:21469


In [36]:
ace_unet_path =  'normattn_sim_fp.har'
get_har_files(ace_unet_path)


['normattn_sim.hn',
 'normattn_sim.native.hn',
 'normattn_sim.npz',
 'normattn_sim.fpo.npz',
 'normattn_sim.stats.npz',
 'normattn_sim.original_model_meta.json',
 'normattn_sim.modifications_meta_data.json',
 'normattn_sim.modification_params.npz',
 'normattn_sim.metadata.json']

In [37]:
params = get_params_from_har(ace_unet_path,params_name='normattn_sim.fpo.npz')
hn = get_params_from_har(ace_unet_path,params_name='normattn_sim.hn')

all_names=['normattn_sim.hn', 'normattn_sim.native.hn', 'normattn_sim.npz', 'normattn_sim.fpo.npz', 'normattn_sim.stats.npz', 'normattn_sim.original_model_meta.json', 'normattn_sim.modifications_meta_data.json', 'normattn_sim.modification_params.npz', 'normattn_sim.metadata.json']
loading  params_name='normattn_sim.fpo.npz' ...
all_names=['normattn_sim.hn', 'normattn_sim.native.hn', 'normattn_sim.npz', 'normattn_sim.fpo.npz', 'normattn_sim.stats.npz', 'normattn_sim.original_model_meta.json', 'normattn_sim.modifications_meta_data.json', 'normattn_sim.modification_params.npz', 'normattn_sim.metadata.json']
loading  params_name='normattn_sim.hn' ...


In [97]:
native_unet_path =  'normattn_sim.har'
get_har_files(native_unet_path)


['normattn_sim.hn',
 'normattn_sim.npz',
 'normattn_sim.original_model_meta.json',
 'normattn_sim.metadata.json']

In [98]:
params_native = get_params_from_har(native_unet_path,params_name='normattn_sim.npz')
hn = get_params_from_har(native_unet_path,params_name='normattn_sim.hn')

all_names=['normattn_sim.hn', 'normattn_sim.npz', 'normattn_sim.original_model_meta.json', 'normattn_sim.metadata.json']
loading  params_name='normattn_sim.npz' ...
all_names=['normattn_sim.hn', 'normattn_sim.npz', 'normattn_sim.original_model_meta.json', 'normattn_sim.metadata.json']
loading  params_name='normattn_sim.hn' ...


In [143]:
ker,bias,kk,bb= get_weight_and_bias_from_layer_name('normattn_sim/conv1',params_native)
ker.shape,bias.shape

((1, 1, 320, 960), (960,))

In [176]:
bias[:10]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [203]:
kn,bn,_,_=get_weight_and_bias_from_layer_name('normattn_sim/normalization1',params_native)
kn.shape,bn.shape

((1, 1, 320, 1), (320,))

In [215]:
calc_snr(norm.weight, kn[0,0,:,0]),calc_snr(norm.bias, bn)

(150.1134741839947, 137.41694237618032)

In [206]:
bn.shape

(320,)

In [212]:
na.attn.to_v.bias#.shape

In [213]:
norm.bias[:10] , bn[:10]

(tensor([ 0.0315, -0.1113,  0.0041,  0.1661, -0.0441,  0.0298,  0.1358,  0.1432,
         -0.0477, -0.0098], grad_fn=<SliceBackward0>),
 array([ 0.03145087, -0.11135001,  0.00409993,  0.16613196, -0.04407737,
         0.02975114,  0.1357694 ,  0.14322889, -0.0477021 , -0.0097683 ],
       dtype=float32))

In [153]:
import onnx
import torch

# Load ONNX model
onnx_model = onnx.load("normattn.sim.onnx")

# Extract weights
weights = {}
for tensor in onnx_model.graph.initializer:
    weights[tensor.name] = torch.tensor(onnx.numpy_helper.to_array(tensor))

In [158]:
weights.keys()

dict_keys(['norm.weight', 'norm.bias', 'attn.to_out.0.bias', 'onnx::MatMul_103', '/attn/Concat_output_0', '/attn/Concat_3_output_0', '_v_64', '_v_59', '/attn/Sqrt_1_output_0'])

In [173]:
weights['_v_64']#[2].item()#.shape#.keys()

tensor([320, 320, 320])

In [174]:
onw = weights['_v_59']#[0]#.shape#.keys()
onw.shape

torch.Size([320, 960])

In [110]:
ker.shape

(1, 1, 320, 960)

In [111]:
na.attn.to_k.weight.shape

torch.Size([320, 320])

In [188]:
calc_snr(ker[0,0,:,:320],na.attn.to_k.weight.data.numpy().transpose(1,0))

-1.1490510014228381

In [175]:
calc_snr(ker[0,0,:,:320],onw[:,:320])

135.1986467287958

In [192]:
na.attn.to_k.weight.data.numpy().transpose(1,0)[1:3,:5]

array([[-0.05896289, -0.15058392,  0.02291918, -0.03649228,  0.07962292],
       [-0.01308832,  0.0248781 , -0.14027154,  0.05371593, -0.01596368]],
      dtype=float32)

In [195]:
ker[0,0,:,:320][1:3,:5].shape

(2, 5)

In [196]:
ker[0,0,:,:320][1:3,:5]/na.attn.to_k.weight.data.numpy().transpose(1,0)[1:3,:5]

array([[ 1.8375612 ,  0.15809733,  2.8913076 ,  0.92508334,  0.08040398],
       [-0.88723475,  2.6247544 , -0.23577859,  1.7135134 , -6.1765523 ]],
      dtype=float32)

In [197]:
ker[0,0,:,:320]/na.attn.to_k.weight.data.numpy().transpose(1,0)

array([[-6.8493176e-01, -6.1119598e-01, -1.8727262e-01, ...,
        -1.6193556e+00,  2.7086648e-01, -5.3612036e-01],
       [ 1.8375612e+00,  1.5809733e-01,  2.8913076e+00, ...,
        -1.1623747e+00, -9.9897838e-01,  1.5679923e+00],
       [-8.8723475e-01,  2.6247544e+00, -2.3577859e-01, ...,
         8.5501510e-01, -1.2034593e+00,  3.2230766e+01],
       ...,
       [ 5.3402658e+00,  1.3224571e+00, -2.9237694e-01, ...,
         8.5732776e-01,  1.1692774e+00, -1.2914703e+00],
       [ 3.8207810e+00,  3.6811090e+00, -2.6521467e-02, ...,
        -1.5882877e+00,  5.4919153e-01,  1.6730946e+00],
       [-7.6752943e-01,  7.1673755e+02,  7.6521957e+01, ...,
         6.1331082e-02,  4.9369395e-01,  2.2270870e+00]], dtype=float32)

In [147]:
bias[320*2:320*3][:10]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [144]:
bias[:320][:10],na.attn.to_k.bias.data.numpy()[:10]

(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32),
 array([ 0.20808674, -0.5583131 , -0.21963541, -0.00398654, -0.13525297,
        -0.06104327,  0.2185896 , -0.18168344, -0.08621909, -0.01317783],
       dtype=float32))

In [199]:
calc_snr(ker[0,0,:,320:320*2],na.attn.to_k.weight.data.numpy().transpose(1,0))

134.45588608263415

In [201]:
calc_snr(ker[0,0,:,:320],na.attn.to_q.weight.data.numpy().transpose(1,0))

135.1986467287958

In [202]:
calc_snr(ker[0,0,:,320*2:],na.attn.to_v.weight.data.numpy().transpose(1,0))

130.47153482535924

In [38]:
uu= UpdateUnet(na, hn,params)

In [39]:
na.attn.to_out


ModuleList(
  (0): Linear(in_features=320, out_features=320, bias=True)
  (1): Dropout(p=0.0, inplace=False)
)

In [43]:
na.norm.weight.shape , na.norm.bias.shape

(torch.Size([320]), torch.Size([320]))

In [53]:
uu.debug = True
uu.update_conv_weights_and_biases('normattn_sim/conv_feature_splitter1_1')
uu.update_conv_weights_and_biases('normattn_sim/conv_feature_splitter1_2')
uu.update_conv_weights_and_biases('normattn_sim/conv_feature_splitter1_3')

layer_name='normattn_sim/conv_feature_splitter1_1' : layer_norm_input=True,ff_input=False,kqv=True,norm.full_name='norm'
attn.full_name='attn',attn.scale=1.0
layer_name='normattn_sim/conv_feature_splitter1_2' : layer_norm_input=True,ff_input=False,kqv=True,norm.full_name='norm'
attn.full_name='attn',attn.scale=1.0
layer_name='normattn_sim/conv_feature_splitter1_3' : layer_norm_input=True,ff_input=False,kqv=True,norm.full_name='norm'
attn.full_name='attn',attn.scale=1.0


In [57]:
calc_snr(na.norm(inp),inter_res['native']['normattn_sim/layer_normalization1'][0][0])

143.4787142297677

In [220]:
out_norm = na.norm(inp)

In [221]:
q= na.attn.to_q(out_norm)
k= na.attn.to_k(out_norm)
v= na.attn.to_v(out_norm)     

In [222]:
calc_snr(k,inter_res['native']['normattn_sim/conv_feature_splitter1_1'][0][0])
calc_snr(q,inter_res['native']['normattn_sim/conv_feature_splitter1_2'][0][0])
calc_snr(v,inter_res['native']['normattn_sim/conv_feature_splitter1_3'][0][0])

73.76657076767458

In [87]:
import torch.nn.functional as F
inner_dim = k.shape[-1]
head_dim = inner_dim // attn.heads
inner_dim

320

In [89]:
ascale,head_dim,(head_dim)**-0.5


(0.15811388300841897, 40, 0.15811388300841897)

In [88]:
key.shape

torch.Size([1, 8, 4096, 40])

In [90]:
query = q.view(1, -1, attn.heads, head_dim).transpose(1, 2)

key = k.view(1, -1, attn.heads, head_dim).transpose(1, 2)
value = v.view(1, -1, attn.heads, head_dim).transpose(1, 2)



        # the output of sdp = (batch, num_heads, seq_len, head_dim)
        # TODO: add support for attn.scale when we move to Torch 2.1
hidden_states = F.scaled_dot_product_attention(
    query, key*ascale, value, attn_mask=None, dropout_p=0.0, is_causal=False
)

hidden_states = hidden_states.transpose(1, 2).reshape(1, -1, attn.heads * head_dim)
hidden_states = hidden_states.to(q.dtype)

In [91]:
calc_snr(hidden_states,inter_res['native']['normattn_sim/matmul2'][0][0])

-2.622365337132555

In [96]:
attn.norm_q, attn.norm_k ,attn.residual_connection, attn.rescale_output_factor

(None, None, False, 1.0)

In [68]:
calc_snr(na(inp),out_org)

-1.4558810326156524

In [67]:
import torch.nn.functional as F

In [ ]:
calc_snr(na.out)

In [228]:
ker,bias,kk,bb= get_weight_and_bias_from_layer_name('normattn_sim/conv2',params_native)
ker.shape,bias.shape

((1, 1, 320, 320), (320,))

In [227]:
na.attn.to_out[0].weight.shape

torch.Size([320, 320])

In [230]:
calc_snr(ker,na.attn.to_out[0].weight.data.numpy().transpose(1,0))

130.06899308201375

In [231]:
calc_snr(bias,na.attn.to_out[0].bias.data.numpy())

129.92092252104203

In [138]:
na.in_channels = 320

In [54]:
na.norm.weight

In [140]:
out_reorg_unet = na(inp)

In [ ]:
from types import MethodType
from qdiff.quant_block import cross_attn_forward
na.attn.forward = MethodType(cross_attn_forward, na.attn)
na.attn.to_out = nn.Sequential(na.attn.to_out[0],na.attn.to_out[1])
na.attn.use_act_quant = False
na.attn.use_weight_quant = False

In [145]:
out_reorg_qnn = na(inp)


In [146]:
calc_snr(out_org,out_reorg_qnn),calc_snr(out_org[0],out_reorg_unet[0])

(1.8657356333999409, 0.7363459764348967)

In [114]:
qnn.model.attn.scale

1.0

In [115]:
qnn.model

normatt(
  (norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
  (attn): Attention(
    (to_q): QuantModule(
      in_features=320, out_features=320, bias=True
      (weight_quantizer): UniformAffineQuantizer(bit=8, scale_method=max, symmetric=False, channel_wise=False, leaf_param=False)
      (act_quantizer): UniformAffineQuantizer(bit=8, scale_method=max, symmetric=False, channel_wise=False, leaf_param=False)
      (activation_function): StraightThrough()
    )
    (to_k): QuantModule(
      in_features=320, out_features=320, bias=True
      (weight_quantizer): UniformAffineQuantizer(bit=8, scale_method=max, symmetric=False, channel_wise=False, leaf_param=False)
      (act_quantizer): UniformAffineQuantizer(bit=8, scale_method=max, symmetric=False, channel_wise=False, leaf_param=False)
      (activation_function): StraightThrough()
    )
    (to_v): QuantModule(
      in_features=320, out_features=320, bias=True
      (weight_quantizer): UniformAffineQuantizer(bit=8, scale_